In [4]:
pip install openai ipywidgets --quiet

In [ ]:
import pandas as pd
import numpy as np
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.tokenize import word_tokenize
import random
import warnings
import openai
from openai import OpenAI
import ipywidgets as widgets
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

# Download the necessary NLTK data package
nltk.download('punkt')
nltk.download('punkt_tab')  # If you really need this for your environment

# Set your OpenAI API key (hardcoded as in original code)
OPENAI_API_KEY = "you api key"
OPENAI_API_URL = "https://api.openai.com/v1/chat/completions"

def load_ebay_data(file_path):
    """Loads the eBay product data from a CSV file."""
    try:
        return pd.read_csv(r"your file path")
    except FileNotFoundError:
        print("Error: Could not find Final_updated_ebay_catalog.csv.")
        return None

def tokenize_titles(df):
    """Tokenizes product titles for improved text processing."""
    df['Tokenized Title'] = df['Title'].astype(str).apply(lambda x: word_tokenize(x.lower()))
    return df

def preprocess_data(df):
    """Vectorizes product titles using TF-IDF and clusters them."""
    df = tokenize_titles(df)
    vectorizer = TfidfVectorizer(stop_words='english')
    X = vectorizer.fit_transform(df['Title'].astype(str))

    num_clusters = min(10, max(2, len(df) // 5))  # Ensuring at least 2 clusters
    model = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    df['Cluster'] = model.fit_predict(X)
    return df, model, vectorizer

# -------- Removed brand references, only price filter remains ---------------
def find_similar_products(df, model, vectorizer, user_query, max_price=None):
    """
    Finds the most similar product titles based on user input using cosine similarity
    and removes duplicates. Now only filters by max price (brand removed).
    """
    query_vec = vectorizer.transform([user_query])
    similarities = cosine_similarity(query_vec, vectorizer.transform(df['Title'].astype(str))).flatten()
    df['Similarity'] = similarities

    # Filter by max price if provided
    if max_price is not None:
        df = df[df['Price'] <= max_price]

    results = df.sort_values(by='Similarity', ascending=False).drop_duplicates(subset=['Title']).head(5)
    return results if not results.empty else None
# ----------------------------------------------------------------------------

def chat_with_openai(messages):
    """Sends multi-turn conversation to OpenAI API and retrieves response."""
    headers = {
        "Authorization": f"Bearer {OPENAI_API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "model": "gpt-3.5-turbo",
        "messages": messages
    }
    try:
        response = requests.post(OPENAI_API_URL, headers=headers, json=data)
        if response.status_code == 200:
            return response.json().get("choices", [{}])[0].get("message", {}).get("content", "Sorry, I couldn't process your request at the moment.")
        else:
            return "Sorry, I couldn't process your request at the moment."
    except requests.exceptions.RequestException as e:
        return f"Error communicating with OpenAI API: {e}"

def chatbot():
    """Enhanced NLP-based chatbot with OpenAI integration and multi-turn conversation."""
    print("\nWelcome to the AI-powered eBay Shopping Chatbot! 🛒")
    print("Choose an option:\n1. Help with your shopping today\n2. Something else")
    user_choice = input("Enter 1 or 2: ").strip()

    # We store some conversation history for GPT usage.
    messages = [{"role": "system", "content": "You are an eBay shopping assistant. You can handle product queries or general chat."}]

    if user_choice == "1":
        df = load_ebay_data("/mnt/data/ebay_cleaned_data.csv")
        df, model, vectorizer = preprocess_data(df)

        while True:
            # user can type 'switch' to go to GPT mode, or 'exit' to quit
            user_query = input("\nWhat product are you looking for? (type 'switch' for general chat, 'exit' to quit): ").strip().lower()
            if user_query == 'exit':
                print("Thanks for using the AI-powered eBay chatbot! 😊")
                break
            if user_query == 'switch':
                # Jump to GPT mode (loop)
                print("\nSwitching to general chat mode...\n")
                while True:
                    gpt_input = input("\nAsk anything or type 'switch' to go back to shopping, 'exit' to quit: ").strip()
                    if gpt_input.lower() == 'exit':
                        print("Thanks for using the AI-powered eBay chatbot! 😊")
                        return  # ends entire chatbot
                    if gpt_input.lower() == 'switch':
                        print("\nSwitching back to shopping mode...\n")
                        break  # break out of GPT loop
                    messages.append({"role": "user", "content": gpt_input})
                    response = chat_with_openai(messages)
                    messages.append({"role": "assistant", "content": response})
                    print(f"\nBot: {response}")
                continue  # resume shopping flow

            try:
                max_price_input = input("Max price? (press enter to skip): ").strip()
                max_price = float(max_price_input) if max_price_input else None
            except ValueError:
                print("Invalid price. Please enter a numeric value.")
                max_price = None

            results = find_similar_products(
                df, model, vectorizer, user_query,
                max_price=max_price
            )

            if results is not None:
                print("\nHere are some matching products for you:")
                for _, row in results.iterrows():
                    print(f"🔹 {row['Title']} - ${row['Price']}")
                    print(f"   Condition: {row['Condition']}")
                    print(f"   Seller: {row['Seller Name']} ({row['Seller Feedback %']}% positive)")
                    print(f"   Product Link: {row['Product URL']}\n")
            else:
                print("\nSorry, I couldn't find any similar products. Try again!")

    elif user_choice == "2":
        # GPT-based loop with a 'switch' to jump to shopping mode.
        while True:
            user_input = input("\nWhat would you like to talk about? (type 'switch' for shopping, 'exit' to quit): ").strip()
            if user_input.lower() == 'exit':
                print("Thanks for chatting! 😊")
                break
            if user_input.lower() == 'switch':
                # Switch to shopping mode
                print("\nSwitching to shopping mode...\n")
                df = load_ebay_data("/mnt/data/ebay_cleaned_data.csv")
                df, model, vectorizer = preprocess_data(df)
                while True:
                    shop_query = input("\nEnter product query or 'switch' for GPT, 'exit' to quit: ").strip().lower()
                    if shop_query == 'exit':
                        print("Thanks for using the AI-powered eBay chatbot! 😊")
                        return
                    if shop_query == 'switch':
                        print("\nSwitching back to general chat mode...\n")
                        break
                    max_price_in = input("Max price? (press enter to skip): ").strip()
                    max_price = float(max_price_in) if max_price_in else None
                    results = find_similar_products(
                        df, model, vectorizer, shop_query,
                        max_price=max_price
                    )
                    if results is not None:
                        print("\nHere are some matching products for you:")
                        for _, row in results.iterrows():
                            print(f"🔹 {row['Title']} - ${row['Price']}")
                            print(f"   Condition: {row['Condition']}")
                            print(f"   Seller: {row['Seller Name']} ({row['Seller Feedback %']}% positive)")
                            print(f"   Product Link: {row['Product URL']}\n")
                    else:
                        print("\nSorry, no similar products. Try again!")
                continue

            # GPT logic
            messages.append({"role": "user", "content": user_input})
            response = chat_with_openai(messages)
            messages.append({"role": "assistant", "content": response})
            print(f"\nBot: {response}")

    else:
        print("Invalid choice. Please restart and enter 1 or 2.")

# Run the chatbot
chatbot()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!



Welcome to the AI-powered eBay Shopping Chatbot! 🛒
Choose an option:
1. Help with your shopping today
2. Something else
Enter 1 or 2: 1

What product are you looking for? (type 'switch' for general chat, 'exit' to quit): switch

Switching to general chat mode...


Ask anything or type 'switch' to go back to shopping, 'exit' to quit: give me a list of red jordan shoes

Bot: Sure! Here are some popular red Jordan shoes available on eBay:

1. Air Jordan 1 Retro High OG "Bred"
2. Air Jordan 11 Retro "Win Like '96"
3. Air Jordan 4 Retro "Toro Bravo"
4. Air Jordan 5 Retro "Red Suede"
5. Air Jordan 13 Retro "Bred"
6. Air Jordan 6 Retro "Infrared"
7. Air Jordan 12 Retro "Gym Red"
8. Air Jordan 3 Retro "Fire Red"
9. Air Jordan 14 Retro "Last Shot"
10. Air Jordan 7 Retro "Raging Bull"

Ask anything or type 'switch' to go back to shopping, 'exit' to quit: give me a list of LV shoes 

Bot: Louis Vuitton (LV) does not manufacture shoes like Nike or Adidas; they exclusively focus on luxury fashion 